## Elering data

In [2]:
import requests
import json
from datetime import datetime

BASE = "https://dashboard.elering.ee/api"
PARAMS = {
    "start": "2026-01-01T00:00:00.000Z",
    "end":   "2026-01-01T02:00:00.000Z"  # kun 2 timer — lille response
}

endpoints = {
    "system":               "/system",
    "system_with_plan":     "/system/with-plan",
    "system_latest":        "/system/latest",
    "flows_hourly":         "/transmission/cross-border/hourly",
    "flows_latest":         "/transmission/cross-border/latest",
    "capacity":             "/transmission/cross-border-capacity",
    "planned_trade":        "/transmission/cross-border-planned-trade",
    "nps_price":            "/nps/price",
    "nps_turnover":         "/nps/turnover",
    "balance":              "/balance",
    "balance_total":        "/balance/total",
    "balance_commerce":     "/balance/commerce",
    "green_certificates":   "/green/certificates",
}

for name, path in endpoints.items():
    try:
        r = requests.get(BASE + path, params=PARAMS)
        data = r.json()
        
        # Hent første datapunkt og print feltnavne
        if isinstance(data, dict) and "data" in data:
            items = data["data"]
            if isinstance(items, list) and len(items) > 0:
                first = items[0]
                print(f"\n{'='*50}")
                print(f" {name} ({path})")
                print(f"   Antal datapunkter: {len(items)}")
                print(f"   Felter: {list(first.keys()) if isinstance(first, dict) else type(first)}")
                print(f"   Første række: {first}")
            elif isinstance(items, dict):
                print(f"\n{'='*50}")
                print(f" {name} ({path})")
                print(f"   Felter: {list(items.keys())}")
                print(f"   Data: {items}")
        else:
            print(f"\n{'='*50}")
            print(f" {name} ({path})")
            print(f"   Rå svar: {str(data)[:200]}")
            
    except Exception as e:
        print(f"\n {name}: {e}")


 system (/system)
   Antal datapunkter: 25
   Felter: ['timestamp', 'production', 'consumption', 'losses', 'frequency', 'system_balance', 'ac_balance', 'production_renewable', 'solar_energy_production']
   Første række: {'timestamp': 1767225600, 'production': 264.18, 'consumption': 1047.28, 'losses': None, 'frequency': 49.96, 'system_balance': -783.1, 'ac_balance': -131.21, 'production_renewable': 76.68, 'solar_energy_production': 4.3}

 system_with_plan (/system/with-plan)
   Felter: ['real', 'plan']
   Data: {'real': [{'timestamp': 1767225600, 'production': 264.18, 'consumption': 1047.28, 'losses': None, 'frequency': 50.0, 'system_balance': -783.1, 'ac_balance': -131.21, 'production_renewable': 76.68, 'solar_energy_production': 4.3}, {'timestamp': 1767226500, 'production': 265.89, 'consumption': 1056.79, 'losses': None, 'frequency': 50.0, 'system_balance': -790.91, 'ac_balance': -122.1, 'production_renewable': 77.83, 'solar_energy_production': 2.8}, {'timestamp': 1767227400, 'produc

## Turnover

In [4]:
import requests
from datetime import datetime, timezone

# Lige før EstLink 2 nedbrud 25. dec 2024
response = requests.get(
    "https://dashboard.elering.ee/api/nps/turnover",
    params={
        "start": "2024-12-25T08:00:00.000Z",
        "end":   "2025-12-25T08:00:00.000Z"
    }
)
data = response.json()
print(data)

{'success': True, 'data': {'ee': [], 'fi': [], 'lv': [], 'lt': []}}


empty

## Print times interval for records

In [26]:
import requests
import pandas as pd

BASE = "https://dashboard.elering.ee/api"
PARAMS = {
    "start": "2026-01-01T00:00:00.000Z",
    "end":   "2026-03-01T00:00:00.000Z"
}

endpoints = {
    "nps_price":        "/nps/price",
    "flows_5min":       "/transmission/cross-border",
    "flows_hourly":     "/transmission/cross-border/hourly",
    "system":           "/system",
    "balance_total":    "/balance/total",
    "balance_detail":   "/balance",
}

for name, ep in endpoints.items():
    r = requests.get(BASE + ep, params=PARAMS)
    resp = r.json()

    if not resp.get("success"):
        print(f"\n{'='*50}\n {name} ({ep})\n   FEJL: {resp}")
        continue

    data = resp["data"]

    # Håndter dict-struktur (fx prices: {'ee': [...], ...})
    if isinstance(data, dict):
        features = list(data.keys())
        print(f"\n{'='*50}\n {name} ({ep})")
        print(f"   Regioner: {features}")
        for region, vals in data.items():
            if vals:
                ts = [v["timestamp"] for v in vals]
                diffs = pd.Series(ts).diff().dropna().unique()
                print(f"   [{region}] {len(vals)} datapunkter — interval: {sorted(diffs)} sek")
            else:
                print(f"   [{region}] 0 datapunkter")

    # Håndter liste-struktur (fx flows, system)
    elif isinstance(data, list) and data:
        features = list(data[0].keys())
        ts = [d["timestamp"] for d in data]
        diffs = pd.Series(ts).diff().dropna().unique()
        null_counts = {f: sum(1 for d in data if d.get(f) is None) for f in features}
        print(f"\n{'='*50}\n {name} ({ep})")
        print(f"   {len(data)} datapunkter — interval: {sorted(diffs)} sek")
        print(f"   Features: {features}")
        print(f"   Manglende værdier (None): { {k:v for k,v in null_counts.items() if v > 0} }")
    else:
        print(f"\n{'='*50}\n {name} ({ep})\n   Ingen data")



 nps_price (/nps/price)
   Regioner: ['ee', 'fi', 'lv', 'lt']
   [ee] 5665 datapunkter — interval: [900.0] sek
   [fi] 5665 datapunkter — interval: [900.0] sek
   [lv] 5665 datapunkter — interval: [900.0] sek
   [lt] 5665 datapunkter — interval: [900.0] sek

 flows_5min (/transmission/cross-border)
   16753 datapunkter — interval: [300.0, 600.0, 2100.0, 3900.0, 36300.0] sek
   Features: ['timestamp', 'finland', 'latvia', 'estlink_1', 'estlink_2', 'russia_narva', 'russia_pihkva']
   Manglende værdier (None): {'latvia': 1, 'estlink_2': 2, 'russia_narva': 16753, 'russia_pihkva': 16753}

 flows_hourly (/transmission/cross-border/hourly)
   1407 datapunkter — interval: [3600.0, 36000.0] sek
   Features: ['timestamp', 'finland', 'latvia', 'estlink_1', 'estlink_2', 'russia_narva', 'russia_pihkva']
   Manglende værdier (None): {'russia_narva': 1407, 'russia_pihkva': 1407}

 system (/system)
   16811 datapunkter — interval: [300.0, 600.0, 2100.0, 3900.0, 36300.0] sek
   Features: ['timestamp',